<a href="https://colab.research.google.com/github/jdeepak-4u/my-new-ai-repo/blob/feature-agent/Problem5_Task_Planning_Agent_Guided_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AgenticOps Hackathon — Problem 5: Task Planning Agent with Iteration

**Stack:** LangGraph · Grok AI · Langfuse

```
generate → evaluate ⇄ improve (min 2 iter) → format → END
```

---

## 🎯 Problem Statement

Build a **Task Planning Agent** that:

| Area | What you will do |
|------|------------------|
| **Build** | Generate an actionable plan (steps / checklist) for any given task |
| **Ops** | Run 3 different inputs, identify weak outputs, improve structure & reasoning |
| **Output** | 3 test cases with iteration improvements |

---

## 🗺️ Agent Flow

```
User Task
    │
    ▼
[ generate_node ]  ──→  Draft a structured plan
    │
    ▼
[ evaluate_node ]  ──→  Score the plan (1-10) + written feedback
    │
    ├── score < threshold OR iterations < MIN  ──→  [ improve_node ]
    │                                                      │
    │                                              (loops back to evaluate)
    │
    └── score ≥ threshold AND iterations ≥ MIN  ──→  [ format_node ]  ──→  END
```

Full prompt + plan content is visible in **Langfuse** at every iteration.


---
## Step 1 — Install Dependencies

Run this cell once. It installs LangGraph, LangChain and Langfuse for observability.

In [1]:
pip install -U langgraph langchain-groq langfuse pydantic python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.2.1
    Uninstalling wrapt-2.2.1:
      Successfully uninstalled wrapt-2.2.1
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
ERROR: pip's dependency res

In [2]:
import os
import getpass

In [3]:

def set_secret_if_missing(env_name: str, required: bool = False, default: str | None = None):
    if os.environ.get(env_name):
        print(f"{env_name}: already set")
        return
    if default is not None:
        os.environ[env_name] = default
        print(f"{env_name}: set to default")
        return
    if required:
        os.environ[env_name] = getpass.getpass(f"Enter {env_name}: ")
    else:
        value = getpass.getpass(f"Enter {env_name} or leave blank to skip: ")
        if value:
            os.environ[env_name] = value

# Required for Groq classifier
set_secret_if_missing("GROQ_API_KEY", required=True)

# Optional Langfuse config
set_secret_if_missing("LANGFUSE_PUBLIC_KEY")
set_secret_if_missing("LANGFUSE_SECRET_KEY")
set_secret_if_missing("LANGFUSE_BASE_URL", default="https://us.cloud.langfuse.com")

print("Configuration cell complete.")

Enter GROQ_API_KEY: ··········
Enter LANGFUSE_PUBLIC_KEY or leave blank to skip: ··········
Enter LANGFUSE_SECRET_KEY or leave blank to skip: ··········
LANGFUSE_BASE_URL: set to default
Configuration cell complete.


In [4]:
GROQ_MODEL = os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile")

---
## Step 2 — Imports & Configuration

**What each piece does:**
- `ChatGroq` — wraps the Grok Hosted llama model
- `StateGraph` — LangGraph's graph builder; each node is a Python function that transforms state
- `Langfuse` — observability layer; logs every prompt, completion, score, and span so you can inspect the full agent run in the UI

**Key thresholds:**
- `SCORE_THRESHOLD = 7.5` — plan must reach this score before the agent can stop
- `MIN_ITERATIONS = 2` — at least 2 improve passes happen regardless of score
- `MAX_ITERATIONS = 3` — hard ceiling; agent formats output after this even if score is low

In [5]:
SCORE_THRESHOLD = 7.5
MIN_ITERATIONS = 2
MAX_ITERATIONS = 3

---
## Step 3 — Define the Agent State

LangGraph passes a **state dictionary** between every node. Think of it as the agent's shared memory across the whole run.

| Field | Type | Purpose |
|-------|------|--------|
| `task` | `str` | The original user task (never changes) |
| `current_plan` | `str` | The plan being worked on (updated each improve pass) |
| `score` | `float` | Latest quality score from the evaluator (1–10) |
| `feedback` | `str` | Written critique from the evaluator |
| `iterations` | `int` | How many improve passes have run so far |
| `score_history` | `list` | All scores collected across iterations |
| `final_plan` | `str` | Formatted final output (set by the format node) |
| `tracer` | `Any` | Langfuse tracer object for observability |

In [6]:
from typing import Any, TypedDict

class AgentState(TypedDict, total=False):
    """
    Shared memory passed between LangGraph nodes.
    Each node reads the current state and returns partial state updates.
    """

    task: str
    current_plan: str
    score: float
    feedback: str
    iterations: int
    score_history: list[float]
    final_plan: str
    tracer: Any

In [36]:
from langchain_groq import ChatGroq
import time
import re
from IPython.display import Markdown, display
generator_llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    max_tokens=None,
    timeout=60,
    max_retries=2,
)

evaluator_llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    max_tokens=None,
    timeout=60,
    max_retries=2,
)

---
## Step 4 — Langfuse Tracer Helper

The `Tracer` class wraps Langfuse calls so every node can log structured data without boilerplate.

- **`.span()`** — logs a named operation with input/output metadata (latency, lengths)
- **`.generation()`** — logs the full prompt **and** completion; this is what you see in the Langfuse *Generations* tab — the most useful view for debugging bad plans
- **`.score()`** — attaches a numeric quality score to the trace so you can filter/compare runs
- **`.done()`** — finalises the trace and flushes to Langfuse

In [23]:
# Step 4 — Langfuse Tracer Helper
from contextlib import contextmanager, nullcontext
from langgraph.graph import StateGraph, START, END

def _jsonable(value: Any) -> str:
    try:
        return json.dumps(value, ensure_ascii=False, default=str)
    except Exception:
        return str(value)


def _char_len(value: Any) -> int:
    return len(_jsonable(value))


class Tracer:
    """
    Thin wrapper over Langfuse.

    Also keeps a local event log in self.local_events so the notebook can build
    weak-output and iteration-improvement tables even if Langfuse credentials
    are not available.
    """

    def __init__(
        self,
        trace_name: str = "task-planning-agent",
        session_id: str | None = None,
        enabled: bool = True,
    ):
        self.trace_name = trace_name
        self.session_id = session_id or str(uuid.uuid4())
        self.local_events: list[dict[str, Any]] = []
        self.client = None

        has_keys = bool(
            os.getenv("LANGFUSE_PUBLIC_KEY")
            and os.getenv("LANGFUSE_SECRET_KEY")
        )

        if enabled and has_keys:
            try:
                self.client = get_client()
            except Exception as exc:
                self.local_events.append(
                    {
                        "type": "tracing_error",
                        "name": "langfuse_init",
                        "error": repr(exc),
                    }
                )
                self.client = None

    def _observation_context(self, **kwargs):
        if self.client is None:
            return nullcontext(None)

        try:
            return self.client.start_as_current_observation(**kwargs)
        except Exception as exc:
            self.local_events.append(
                {
                    "type": "tracing_error",
                    "name": "start_observation",
                    "error": repr(exc),
                    "kwargs": {k: v for k, v in kwargs.items() if k != "input"},
                }
            )
            return nullcontext(None)

    @contextmanager
    def span(
        self,
        name: str,
        input: Any | None = None,
        metadata: dict[str, Any] | None = None,
    ):
        metadata = metadata or {}
        started = time.perf_counter()

        event = {
            "type": "span",
            "name": name,
            "input": input,
            "output": None,
            "metadata": metadata,
            "input_chars": _char_len(input),
            "output_chars": 0,
            "latency_ms": None,
            "status": "running",
        }
        self.local_events.append(event)

        with self._observation_context(
            name=name,
            as_type="span",
            input=input,
            metadata={
                **metadata,
                "trace_name": self.trace_name,
                "session_id": self.session_id,
            },
        ) as obs:
            try:
                yield event
                event["status"] = "ok"
            except Exception as exc:
                event["status"] = "error"
                event["error"] = repr(exc)
                raise
            finally:
                event["latency_ms"] = round((time.perf_counter() - started) * 1000, 2)
                event["output_chars"] = _char_len(event.get("output"))

                if obs is not None:
                    try:
                        obs.update(
                            output=event.get("output"),
                            metadata={
                                **metadata,
                                "latency_ms": event["latency_ms"],
                                "input_chars": event["input_chars"],
                                "output_chars": event["output_chars"],
                            },
                        )
                    except Exception as exc:
                        event["trace_update_error"] = repr(exc)

    @contextmanager
    def generation(
        self,
        name: str,
        prompt: Any,
        model: str,
        metadata: dict[str, Any] | None = None,
    ):
        metadata = metadata or {}
        started = time.perf_counter()

        event = {
            "type": "generation",
            "name": name,
            "model": model,
            "input": prompt,
            "output": None,
            "metadata": metadata,
            "input_chars": _char_len(prompt),
            "output_chars": 0,
            "latency_ms": None,
            "status": "running",
        }
        self.local_events.append(event)

        with self._observation_context(
            name=name,
            as_type="generation",
            model=model,
            input=prompt,
            metadata={
                **metadata,
                "trace_name": self.trace_name,
                "session_id": self.session_id,
            },
        ) as obs:
            try:
                yield event
                event["status"] = "ok"
            except Exception as exc:
                event["status"] = "error"
                event["error"] = repr(exc)
                raise
            finally:
                event["latency_ms"] = round((time.perf_counter() - started) * 1000, 2)
                event["output_chars"] = _char_len(event.get("output"))

                if obs is not None:
                    try:
                        obs.update(
                            output=event.get("output"),
                            metadata={
                                **metadata,
                                "latency_ms": event["latency_ms"],
                                "input_chars": event["input_chars"],
                                "output_chars": event["output_chars"],
                                "usage_metadata": event.get("usage_metadata", {}),
                            },
                        )
                    except Exception as exc:
                        event["trace_update_error"] = repr(exc)

    def score(
        self,
        name: str,
        value: float,
        comment: str = "",
        metadata: dict[str, Any] | None = None,
    ):
        metadata = metadata or {}

        event = {
            "type": "score",
            "name": name,
            "value": float(value),
            "comment": comment,
            "metadata": metadata,
        }
        self.local_events.append(event)

        if self.client is not None:
            try:
                self.client.score_current_trace(
                    name=name,
                    value=float(value),
                    data_type="NUMERIC",
                    comment=comment[:500],
                    metadata=metadata,
                )
            except Exception as exc:
                event["trace_score_error"] = repr(exc)

    def done(self):
        if self.client is not None:
            try:
                self.client.flush()
            except Exception as exc:
                self.local_events.append(
                    {
                        "type": "tracing_error",
                        "name": "langfuse_flush",
                        "error": repr(exc),
                    }
                )

---
## Step 5 — Define the Four Agent Nodes

Each node is a **pure Python function** that receives the current state and returns an updated state.

### Node 1 — `generate_node`
Sends the raw task to the LLM with a structured prompt that forces five sections:  
`Goal → Prerequisites → Steps → Risks → Success Criteria`  
This gives the evaluator something concrete to score rather than freeform prose.

### Node 2 — `evaluate_node`
Sends the current plan to the **evaluator LLM** and asks it to return `{"score": float, "feedback": str}`.  
Uses a 3-layer JSON parse with a safe fallback so the pipeline never crashes on a malformed response.

### Node 3 — `improve_node`
Sends the evaluator's feedback **plus** the current plan back to the generator LLM with an explicit instruction to fix identified issues while keeping the same section structure.

### Node 4 — `format_node`
Assembles the final Markdown output with a score bar, quality label, and the full score journey table.

In [28]:
# Step 5 — Define the Four Agent Nodes

GENERATOR_SYSTEM = """
You are a Task Planning Agent for Build and Ops work.

Create a practical, actionable plan in Markdown with exactly these five sections:

## Goal
## Prerequisites
## Steps
## Risks
## Success Criteria

Rules:
- Make the plan operational, not generic.
- Every major step must include an owner role, action, evidence/artifact, and sequencing.
- For Ops tasks, include verification checks, rollback/escalation logic, and weak-output detection.
- Avoid hidden chain-of-thought. Give concise rationale only where useful.
"""


EVALUATOR_SYSTEM = """
You are a strict evaluator for task plans.

Return only valid JSON with this exact schema:
{
  "score": 7.0,
  "feedback": "Weak outputs:\\n- ...\\nImprovement guidance:\\n- ..."
}

Score from 1 to 10 using:
- Actionability: owners, actions, evidence.
- Structure: exact five sections are present.
- Sequencing: clear order, milestones, dependencies.
- Reasoning: assumptions and tradeoffs are clear.
- Risk control: risks, rollback, mitigation, verification.
- Ops quality: weak-output detection, validation, escalation, evidence.

Be critical. Identify weak outputs even if the plan is decent.
Do not include markdown fences.
Do not reveal hidden chain-of-thought.
"""


IMPROVER_SYSTEM = """
You are an expert plan improver.

Improve the current plan using the evaluator feedback.

Requirements:
- Keep exactly the same five Markdown sections:
  Goal, Prerequisites, Steps, Risks, Success Criteria.
- Fix vague steps.
- Add missing owners, evidence, sequencing, validation, rollback, risks, and success metrics.
- For Ops tasks, strengthen weak-output detection and verification.
- Do not simply rephrase; make the plan more operational.
- Do not reveal hidden chain-of-thought.
"""


def _messages_to_prompt(messages: list[tuple[str, str]]) -> str:
    return "\n\n".join([f"{role.upper()}:\n{content}" for role, content in messages])


def call_llm(
    llm: ChatGroq,
    messages: list[tuple[str, str]],
    tracer: Tracer,
    generation_name: str,
    metadata: dict[str, Any] | None = None,
) -> str:
    prompt_text = _messages_to_prompt(messages)

    with tracer.generation(
        name=generation_name,
        prompt=prompt_text,
        model=GROQ_MODEL,
        metadata=metadata or {},
    ) as gen:
        response = llm.invoke(messages)
        content = getattr(response, "content", str(response))
        gen["output"] = content
        gen["usage_metadata"] = getattr(response, "usage_metadata", {})

    return content.strip()


def _extract_first_json_object(text: str) -> str | None:
    fenced = re.search(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if fenced:
        return fenced.group(1)

    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        char = text[i]

        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    return text[start : i + 1]

    return None


def _clamp_score(value: Any) -> float:
    try:
        score = float(value)
    except Exception:
        score = 5.0

    # Handles accidental 0–100 scale.
    if 10 < score <= 100:
        score = score / 10

    return round(max(1.0, min(10.0, score)), 2)


def parse_evaluation_json(raw_text: str) -> dict[str, Any]:
    """
    3-layer JSON parse:
    1. Direct json.loads(raw_text)
    2. Extract first JSON object, then json.loads(...)
    3. Regex fallback for score + raw feedback
    """

    candidates = [raw_text, _extract_first_json_object(raw_text)]

    for candidate in candidates:
        if not candidate:
            continue

        try:
            parsed = json.loads(candidate)
            return {
                "score": _clamp_score(parsed.get("score", 5.0)),
                "feedback": str(parsed.get("feedback", "No feedback returned.")),
            }
        except Exception:
            pass

    score_match = re.search(
        r"(?i)(?:score|rating)\D{0,30}(\d+(?:\.\d+)?)",
        raw_text,
    )

    score = _clamp_score(score_match.group(1)) if score_match else 5.0

    feedback_match = re.search(
        r"(?is)feedback\s*[:\-]\s*(.*)",
        raw_text,
    )

    feedback = (
        feedback_match.group(1).strip()
        if feedback_match
        else f"Evaluator returned malformed JSON. Raw response excerpt: {raw_text[:1200]}"
    )

    return {
        "score": score,
        "feedback": feedback,
    }


def score_bar(score: float, width: int = 10) -> str:
    filled = round((score / 10) * width)
    return f"{'█' * filled}{'░' * (width - filled)} {score:.1f}/10"


def quality_label(score: float) -> str:
    if score >= 8.5:
        return "Excellent"
    if score >= SCORE_THRESHOLD:
        return "Pass"
    if score >= 6.0:
        return "Needs Improvement"
    return "Weak"


def score_journey_table(score_history: list[float]) -> str:
    rows = [
        "| Evaluation Round | Score | Delta | Status |",
        "|---|---:|---:|---|",
    ]

    for idx, score in enumerate(score_history):
        round_name = "Initial" if idx == 0 else f"After Improve Pass {idx}"
        delta = "—" if idx == 0 else f"{score - score_history[idx - 1]:+.2f}"
        status = "Pass" if score >= SCORE_THRESHOLD else "Below Threshold"
        rows.append(f"| {round_name} | {score:.2f} | {delta} | {status} |")

    return "\n".join(rows)


def generate_node(state: AgentState) -> dict[str, Any]:
    tracer: Tracer = state["tracer"]

    with tracer.span(
        "generate_node",
        input={"task": state["task"]},
        metadata={"iterations": state.get("iterations", 0)},
    ) as span:
        messages = [
            ("system", GENERATOR_SYSTEM),
            (
                "human",
                f"""
Create the initial plan for this task:

{state["task"]}

Remember: use exactly the five required sections.
""",
            ),
        ]

        plan = call_llm(
            llm=generator_llm,
            messages=messages,
            tracer=tracer,
            generation_name="generate_initial_plan",
            metadata={"node": "generate_node"},
        )

        span["output"] = {
            "plan_chars": len(plan),
            "section_check": {
                "Goal": "## Goal" in plan,
                "Prerequisites": "## Prerequisites" in plan,
                "Steps": "## Steps" in plan,
                "Risks": "## Risks" in plan,
                "Success Criteria": "## Success Criteria" in plan,
            },
        }

        return {
            "current_plan": plan,
        }


def evaluate_node(state: AgentState) -> dict[str, Any]:
    tracer: Tracer = state["tracer"]
    current_iteration = state.get("iterations", 0)

    with tracer.span(
        "evaluate_node",
        input={
            "task": state["task"],
            "iterations": current_iteration,
            "current_plan": state["current_plan"],
        },
        metadata={"iterations": current_iteration},
    ) as span:
        messages = [
            ("system", EVALUATOR_SYSTEM),
            (
                "human",
                f"""
Task:
{state["task"]}

Current plan:
{state["current_plan"]}

Evaluate the plan and return JSON only.
""",
            ),
        ]

        raw_eval = call_llm(
            llm=evaluator_llm,
            messages=messages,
            tracer=tracer,
            generation_name=f"evaluate_plan_iter_{current_iteration}",
            metadata={"node": "evaluate_node", "iterations": current_iteration},
        )

        parsed = parse_evaluation_json(raw_eval)

        score = parsed["score"]
        feedback = parsed["feedback"]

        score_history = list(state.get("score_history", []))
        score_history.append(score)

        tracer.score(
            name=f"plan_quality_iter_{current_iteration}",
            value=score,
            comment=feedback,
            metadata={"iterations": current_iteration},
        )

        span["output"] = {
            "iterations": current_iteration,
            "score": score,
            "feedback": feedback,
            "score_history": score_history,
            "raw_eval_excerpt": raw_eval[:1200],
        }

        return {
            "score": score,
            "feedback": feedback,
            "score_history": score_history,
        }


def improve_node(state: AgentState) -> dict[str, Any]:
    tracer: Tracer = state["tracer"]
    next_iteration = state.get("iterations", 0) + 1

    with tracer.span(
        "improve_node",
        input={
            "task": state["task"],
            "next_iteration": next_iteration,
            "score": state.get("score"),
            "feedback": state.get("feedback"),
            "current_plan": state.get("current_plan"),
        },
        metadata={"next_iteration": next_iteration},
    ) as span:
        messages = [
            ("system", IMPROVER_SYSTEM),
            (
                "human",
                f"""
Task:
{state["task"]}

Current score:
{state.get("score", 0)}

Evaluator feedback:
{state.get("feedback", "")}

Current plan:
{state.get("current_plan", "")}

Create the improved plan for improve pass {next_iteration}.
""",
            ),
        ]

        improved_plan = call_llm(
            llm=generator_llm,
            messages=messages,
            tracer=tracer,
            generation_name=f"improve_plan_iter_{next_iteration}",
            metadata={"node": "improve_node", "next_iteration": next_iteration},
        )

        span["output"] = {
            "next_iteration": next_iteration,
            "improved_plan_chars": len(improved_plan),
        }

        return {
            "current_plan": improved_plan,
            "iterations": next_iteration,
        }


def format_node(state: AgentState) -> dict[str, Any]:
    tracer: Tracer = state["tracer"]
    score = float(state.get("score", 0.0))
    history = state.get("score_history", [])

    with tracer.span(
        "format_node",
        input={
            "score": score,
            "iterations": state.get("iterations", 0),
            "score_history": history,
        },
        metadata={"iterations": state.get("iterations", 0)},
    ) as span:
        if state.get("iterations", 0) >= MAX_ITERATIONS and score < SCORE_THRESHOLD:
            stop_reason = "MAX_ITERATIONS reached before threshold"
        elif state.get("iterations", 0) >= MIN_ITERATIONS and score >= SCORE_THRESHOLD:
            stop_reason = "Threshold met after minimum improve passes"
        else:
            stop_reason = "Formatted by router"

        final_plan = f"""
# Task Planning Agent — Final Output

**Quality:** {quality_label(score)}
**Final Score:** {score_bar(score)}
**Improve Passes Completed:** {state.get("iterations", 0)}
**Stop Reason:** {stop_reason}

---

## Score Journey

{score_journey_table(history)}

---

## Final Evaluator Feedback

{state.get("feedback", "")}

---

## Final Plan

{state.get("current_plan", "")}
""".strip()

        span["output"] = {
            "final_score": score,
            "quality_label": quality_label(score),
            "iterations": state.get("iterations", 0),
            "stop_reason": stop_reason,
            "final_plan_chars": len(final_plan),
        }

        return {
            "final_plan": final_plan,
        }

---
## Step 6 — Routing Logic & Graph Assembly

The `route()` function is the **decision point** after every evaluation. It implements three rules in priority order:

1. **Force improve** if `iterations < MIN_ITERATIONS` (ensures at least 2 passes regardless of score)
2. **Finalize** if `score >= SCORE_THRESHOLD` OR `iterations >= MAX_ITERATIONS` (good enough, or hard ceiling reached)
3. **Improve** otherwise (score still too low and budget remains)

LangGraph wires these together as a **conditional edge** on the `evaluate` node.

In [29]:
# Step 6 — Routing Logic & Graph Assembly

def route_after_evaluation(state: AgentState) -> str:
    """
    Enforces:
    - At least 2 improve passes.
    - Continue improving if score < threshold and iterations < max.
    - Stop after max iterations even if score is still low.
    """

    iterations = state.get("iterations", 0)
    score = float(state.get("score", 0.0))

    if iterations < MIN_ITERATIONS:
        return "improve"

    if score < SCORE_THRESHOLD and iterations < MAX_ITERATIONS:
        return "improve"

    return "format"


graph_builder = StateGraph(AgentState)

graph_builder.add_node("generate", generate_node)
graph_builder.add_node("evaluate", evaluate_node)
graph_builder.add_node("improve", improve_node)
graph_builder.add_node("format", format_node)

graph_builder.add_edge(START, "generate")
graph_builder.add_edge("generate", "evaluate")

graph_builder.add_conditional_edges(
    "evaluate",
    route_after_evaluation,
    {
        "improve": "improve",
        "format": "format",
    },
)

graph_builder.add_edge("improve", "evaluate")
graph_builder.add_edge("format", END)

APP = graph_builder.compile()

---
## Step 7 — Runner Function

`run()` wires together the tracer, the initial state, and the graph invocation for a single task.  
It returns the full final state so you can inspect individual fields (score, score_history, final_plan) programmatically after the run.

In [30]:
# Step 7 — Runner Function

def run_task_planner(
    task: str,
    case_id: str = "manual",
    tracing_enabled: bool = True,
) -> AgentState:
    tracer = Tracer(
        trace_name=f"task-planning-agent:{case_id}",
        session_id=case_id,
        enabled=tracing_enabled,
    )

    initial_state: AgentState = {
        "task": task,
        "current_plan": "",
        "score": 0.0,
        "feedback": "",
        "iterations": 0,
        "score_history": [],
        "final_plan": "",
        "tracer": tracer,
    }

    with tracer.span(
        "agent_run",
        input={
            "case_id": case_id,
            "task": task,
            "score_threshold": SCORE_THRESHOLD,
            "min_iterations": MIN_ITERATIONS,
            "max_iterations": MAX_ITERATIONS,
        },
        metadata={
            "case_id": case_id,
            "model": GROQ_MODEL,
        },
    ) as root_span:
        result = APP.invoke(initial_state)

        root_span["output"] = {
            "case_id": case_id,
            "final_score": result.get("score"),
            "iterations": result.get("iterations"),
            "score_history": result.get("score_history"),
            "final_plan_chars": len(result.get("final_plan", "")),
        }

        tracer.score(
            name="final_plan_quality",
            value=float(result.get("score", 0.0)),
            comment=result.get("feedback", ""),
            metadata={
                "case_id": case_id,
                "iterations": result.get("iterations"),
            },
        )

    tracer.done()
    return result

---
## Step 8 — Run the 3 Test Cases  *(Ops Task)*

The three tasks below cover different planning domains — product launch, ML engineering, and event management.  
Running all three lets you compare how the agent handles varied complexity and terminology.

**What to observe per run:**
- Does the initial score vary by task domain?
- How many iterations does each task need before meeting the threshold?
- Does the score always increase between iterations, or can it dip?

After all three runs, open **Langfuse** and compare traces side-by-side using the Generations tab.

In [33]:
# Step 8 — Run the 3 Test Cases: Ops Task

import pandas as pd

OPS_TEST_CASES = [
    {
        "id": "OPS-001-API-LATENCY-RUNBOOK",
        "task": """
Ops Task:
Create an incident runbook for API latency spikes.

The plan must identify weak outputs and improve structure/reasoning across iterations.
Include detection, triage, owner roles, escalation, rollback, customer comms,
verification evidence, and post-incident review.
""".strip(),
    },
    {
        "id": "OPS-002-PAYMENT-WEBHOOKS",
        "task": """
Ops Task:
Create an operations checklist for failed payment webhooks.

The plan must identify weak outputs and improve structure/reasoning across iterations.
Include alert validation, replay strategy, duplicate prevention, owner roles,
customer-impact checks, rollback criteria, and audit evidence.
""".strip(),
    },
    {
        "id": "OPS-003-KPI-AUTOMATION",
        "task": """
Ops Task:
Create a weekly KPI reporting automation plan.

The plan must identify weak outputs and improve structure/reasoning across iterations.
Include data freshness checks, transformation validation, owner roles,
failure handling, report delivery, stakeholder signoff, and quality metrics.
""".strip(),
    },
]


ops_results: list[dict[str, Any]] = []

for case in OPS_TEST_CASES:
    print(f"Running {case['id']}...")
    final_state = run_task_planner(
        task=case["task"],
        case_id=case["id"],
        tracing_enabled=True,
    )

    ops_results.append(
        {
            "id": case["id"],
            "task": case["task"],
            "state": final_state,
        }
    )


summary_rows = []

for item in ops_results:
    state = item["state"]
    score_history = state.get("score_history", [])

    summary_rows.append(
        {
            "test_case": item["id"],
            "initial_score": score_history[0] if score_history else None,
            "final_score": state.get("score"),
            "improve_passes": state.get("iterations"),
            "score_history": " → ".join([f"{s:.2f}" for s in score_history]),
            "quality": quality_label(float(state.get("score", 0.0))),
        }
    )

ops_summary_df = pd.DataFrame(summary_rows)
display(ops_summary_df)

Running OPS-001-API-LATENCY-RUNBOOK...
Running OPS-002-PAYMENT-WEBHOOKS...
Running OPS-003-KPI-AUTOMATION...


,test_case,initial_score,final_score,improve_passes,score_history,quality
0,OPS-001-API-LATENCY-RUNBOOK,8.5,8.5,2,8.50 → 8.50 → 8.50,Excellent
1,OPS-002-PAYMENT-WEBHOOKS,8.5,8.5,2,8.50 → 8.50 → 8.50,Excellent
2,OPS-003-KPI-AUTOMATION,8.5,8.5,2,8.50 → 8.50 → 8.50,Excellent


---
## Step 9 — Identify Weak Outputs  *(Ops Task)*

After the three runs, examine the results to find which outputs were weakest.  
This cell prints a summary table so you can quickly spot low-scoring or slow-converging plans.

**Signs of a weak output:**
- Final score below 7.5 (hit `MAX_ITERATIONS` before reaching threshold)
- Large drop in score between iterations (evaluator found regressions)
- Flat score history (improve node is not addressing the feedback)
- Very short `plan_length` in spans (plan lacks detail)

In [34]:
# Step 9 — Identify Weak Outputs: Ops Task

WEAK_KEYWORDS = (
    "missing",
    "vague",
    "unclear",
    "weak",
    "lacks",
    "lack",
    "absent",
    "insufficient",
    "generic",
    "not specified",
    "no ",
    "without",
)


def extract_evaluation_events(state: AgentState) -> list[dict[str, Any]]:
    tracer: Tracer = state["tracer"]

    return [
        event
        for event in tracer.local_events
        if event.get("type") == "span" and event.get("name") == "evaluate_node"
    ]


def extract_weak_outputs(feedback: str) -> list[str]:
    if not feedback:
        return []

    lower_feedback = feedback.lower()
    start_idx = lower_feedback.find("weak outputs")
    end_idx = lower_feedback.find("improvement guidance")

    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        section = feedback[start_idx:end_idx]
    elif start_idx != -1:
        section = feedback[start_idx:]
    else:
        section = feedback

    weak_items = []

    for line in section.splitlines():
        cleaned = line.strip().lstrip("-*•0123456789. ").strip()

        if not cleaned:
            continue

        if any(keyword in cleaned.lower() for keyword in WEAK_KEYWORDS):
            weak_items.append(cleaned)

    if weak_items:
        return weak_items

    # Fallback: sentence-level extraction.
    sentences = re.split(r"(?<=[.!?])\s+", feedback.strip())

    for sentence in sentences:
        if any(keyword in sentence.lower() for keyword in WEAK_KEYWORDS):
            weak_items.append(sentence.strip())

    return weak_items[:8]


weak_output_rows = []

for item in ops_results:
    eval_events = extract_evaluation_events(item["state"])

    for eval_round, event in enumerate(eval_events):
        output = event.get("output", {}) or {}
        feedback = output.get("feedback", "")
        score = output.get("score")

        weak_outputs = extract_weak_outputs(feedback)

        if not weak_outputs:
            weak_output_rows.append(
                {
                    "test_case": item["id"],
                    "evaluation_round": eval_round,
                    "score": score,
                    "weak_output": "No weak output extracted from evaluator feedback.",
                }
            )
        else:
            for weak_output in weak_outputs:
                weak_output_rows.append(
                    {
                        "test_case": item["id"],
                        "evaluation_round": eval_round,
                        "score": score,
                        "weak_output": weak_output,
                    }
                )

weak_outputs_df = pd.DataFrame(weak_output_rows)
display(weak_outputs_df)

,test_case,evaluation_round,score,weak_output
0,OPS-001-API-LATENCY-RUNBOOK,0,8.5,Weak outputs:
1,OPS-001-API-LATENCY-RUNBOOK,0,8.5,Lack of specific metrics for measuring the eff...
2,OPS-001-API-LATENCY-RUNBOOK,0,8.5,Insufficient detail on the criteria for escala...
3,OPS-001-API-LATENCY-RUNBOOK,0,8.5,Limited discussion on the verification of evid...
4,OPS-001-API-LATENCY-RUNBOOK,1,8.5,Weak outputs:\n- The plan assumes that the mon...
5,OPS-001-API-LATENCY-RUNBOOK,2,8.5,Weak outputs:\n- Potential for inadequate docu...
6,OPS-002-PAYMENT-WEBHOOKS,0,8.5,Weak outputs:
7,OPS-002-PAYMENT-WEBHOOKS,0,8.5,Lack of specific metrics for measuring the eff...
8,OPS-002-PAYMENT-WEBHOOKS,0,8.5,Insufficient detail on escalation procedures a...
9,OPS-002-PAYMENT-WEBHOOKS,0,8.5,No clear definition of the iterative review pr...


---
## Step 10 — Iteration Improvements Analysis  *(Output Required)*

This cell produces a per-task iteration breakdown — the second required output.

For each test case you can see how the plan evolved across passes, making it easy to write your analysis of what improved (structure, reasoning, specificity).

In [37]:
# Step 10 — Iteration Improvements Analysis: Output Required

def build_iteration_journey_df(results: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []

    for item in results:
        state = item["state"]
        history = state.get("score_history", [])

        for idx, score in enumerate(history):
            rows.append(
                {
                    "test_case": item["id"],
                    "round": "Initial" if idx == 0 else f"After Improve {idx}",
                    "score": score,
                    "delta_from_previous": None if idx == 0 else round(score - history[idx - 1], 2),
                    "threshold_met": score >= SCORE_THRESHOLD,
                }
            )

    return pd.DataFrame(rows)


def build_iteration_summary_df(results: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []

    for item in results:
        state = item["state"]
        history = state.get("score_history", [])

        row = {
            "test_case": item["id"],
            "initial_score": history[0] if len(history) > 0 else None,
            "after_improve_1": history[1] if len(history) > 1 else None,
            "after_improve_2": history[2] if len(history) > 2 else None,
            "after_improve_3": history[3] if len(history) > 3 else None,
            "final_score": history[-1] if history else None,
            "total_gain": round(history[-1] - history[0], 2) if len(history) >= 2 else None,
            "improve_passes_completed": state.get("iterations"),
            "quality": quality_label(float(state.get("score", 0.0))),
        }

        rows.append(row)

    return pd.DataFrame(rows)


def build_required_output_report(results: list[dict[str, Any]]) -> str:
    parts = [
        "# Required Output — 3 Test Cases with Iteration Improvements",
        "",
        f"Threshold: **{SCORE_THRESHOLD}**  ",
        f"Minimum improve passes: **{MIN_ITERATIONS}**  ",
        f"Maximum improve passes: **{MAX_ITERATIONS}**",
        "",
    ]

    for item in results:
        state = item["state"]
        history = state.get("score_history", [])
        eval_events = extract_evaluation_events(state)

        initial_feedback = ""
        final_feedback = ""

        if eval_events:
            initial_feedback = (eval_events[0].get("output", {}) or {}).get("feedback", "")
            final_feedback = (eval_events[-1].get("output", {}) or {}).get("feedback", "")

        initial_weak_outputs = extract_weak_outputs(initial_feedback)[:5]
        final_weak_outputs = extract_weak_outputs(final_feedback)[:5]

        score_path = " → ".join([f"{score:.2f}" for score in history])
        total_gain = history[-1] - history[0] if len(history) >= 2 else 0.0

        parts.extend(
            [
                f"## {item['id']}",
                "",
                f"**Score journey:** `{score_path}`  ",
                f"**Total improvement:** `{total_gain:+.2f}`  ",
                f"**Improve passes completed:** `{state.get('iterations')}`  ",
                f"**Final quality:** `{quality_label(float(state.get('score', 0.0)))}`",
                "",
                "**Initial weak outputs detected:**",
            ]
        )

        if initial_weak_outputs:
            parts.extend([f"- {issue}" for issue in initial_weak_outputs])
        else:
            parts.append("- No initial weak output extracted.")

        parts.extend(
            [
                "",
                "**Remaining weak outputs after final evaluation:**",
            ]
        )

        if final_weak_outputs:
            parts.extend([f"- {issue}" for issue in final_weak_outputs])
        else:
            parts.append("- No remaining weak output extracted.")

        parts.append("")

    return "\n".join(parts)


iteration_journey_df = build_iteration_journey_df(ops_results)
iteration_summary_df = build_iteration_summary_df(ops_results)

display(Markdown("## Iteration Journey Table"))
display(iteration_journey_df)

display(Markdown("## 3 Test Cases — Iteration Improvement Summary"))
display(iteration_summary_df)

display(Markdown(build_required_output_report(ops_results)))

## Iteration Journey Table

,test_case,round,score,delta_from_previous,threshold_met
0,OPS-001-API-LATENCY-RUNBOOK,Initial,8.5,NaN,True
1,OPS-001-API-LATENCY-RUNBOOK,After Improve 1,8.5,0.0,True
2,OPS-001-API-LATENCY-RUNBOOK,After Improve 2,8.5,0.0,True
3,OPS-002-PAYMENT-WEBHOOKS,Initial,8.5,NaN,True
4,OPS-002-PAYMENT-WEBHOOKS,After Improve 1,8.5,0.0,True
5,OPS-002-PAYMENT-WEBHOOKS,After Improve 2,8.5,0.0,True
6,OPS-003-KPI-AUTOMATION,Initial,8.5,NaN,True
7,OPS-003-KPI-AUTOMATION,After Improve 1,8.5,0.0,True
8,OPS-003-KPI-AUTOMATION,After Improve 2,8.5,0.0,True


## 3 Test Cases — Iteration Improvement Summary

,test_case,initial_score,after_improve_1,after_improve_2,after_improve_3,final_score,total_gain,improve_passes_completed,quality
0,OPS-001-API-LATENCY-RUNBOOK,8.5,8.5,8.5,None,8.5,0.0,2,Excellent
1,OPS-002-PAYMENT-WEBHOOKS,8.5,8.5,8.5,None,8.5,0.0,2,Excellent
2,OPS-003-KPI-AUTOMATION,8.5,8.5,8.5,None,8.5,0.0,2,Excellent


# Required Output — 3 Test Cases with Iteration Improvements

Threshold: **7.5**  
Minimum improve passes: **2**  
Maximum improve passes: **3**

## OPS-001-API-LATENCY-RUNBOOK

**Score journey:** `8.50 → 8.50 → 8.50`  
**Total improvement:** `+0.00`  
**Improve passes completed:** `2`  
**Final quality:** `Excellent`

**Initial weak outputs detected:**
- Weak outputs:
- Lack of specific metrics for measuring the effectiveness of mitigations and the overall incident response process.
- Insufficient detail on the criteria for escalating issues to engineering leadership and the process for broader system adjustments.
- Limited discussion on the verification of evidence and artifacts, particularly in the context of weak output detection and validation.

**Remaining weak outputs after final evaluation:**
- Weak outputs:\n- Potential for inadequate documentation or misunderstanding of system dependencies complicating triage, despite regular documentation updates and dependency map reviews.\n- Mitigations causing unintended side effects or not addressing the root cause, despite thorough mitigation planning and continuous monitoring.\n- Communication breakdowns leading to stakeholder dissatisfaction, despite regular communication checks and stakeholder feedback surveys.\n

## OPS-002-PAYMENT-WEBHOOKS

**Score journey:** `8.50 → 8.50 → 8.50`  
**Total improvement:** `+0.00`  
**Improve passes completed:** `2`  
**Final quality:** `Excellent`

**Initial weak outputs detected:**
- Weak outputs:
- Lack of specific metrics for measuring the effectiveness of alert validation and replay strategy.
- Insufficient detail on escalation procedures and communication strategies for affected customers.
- No clear definition of the iterative review process for identifying and addressing weak outputs.

**Remaining weak outputs after final evaluation:**
- Weak outputs: \n- The plan does not explicitly address the potential impact of external factors, such as network outages or third-party service disruptions, on the payment webhook process. \n- There is a lack of detailed information on the specific metrics used to measure the effectiveness of the duplicate prevention measures. \n

## OPS-003-KPI-AUTOMATION

**Score journey:** `8.50 → 8.50 → 8.50`  
**Total improvement:** `+0.00`  
**Improve passes completed:** `2`  
**Final quality:** `Excellent`

**Initial weak outputs detected:**
- Weak outputs:
- Insufficient details on ETL script failure or data transformation error mitigation,

**Remaining weak outputs after final evaluation:**
- Weak outputs:\n- Inadequate handling of data source connectivity issues or access permission problems in the event of prolonged outages or security breaches\n- Limited discussion on the implementation of data quality checks and validation for transformed data\n- Insufficient detail on the metrics used to measure report quality and stakeholder satisfaction\n


---
*AgenticOps Hackathon · Problem 5 · LangGraph · Grok AI · Langfuse*